## Data exploration and patient-level validation splits

Prepares a stratified five-fold cross-validation split where:

- `class_id`1-3 -> IDHmutant
- `class_id`4-6 -> IDHwildtype

The dataset contains sample entries, each with a 3D Raman array. A patient can have multiple entries, and each entry contains many spectra. The patient is therefore the split unit: and every entry and spectrum from one patient must stay in the same fold.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold

PATH = r"..\data\preprocessed.npy"
dataset = np.load(PATH, allow_pickle=True)

print(f"Sample entries: {len(dataset)}")

Sample entries: 59


### Sample-level metadata table

Each top-level dataset item is treated as one sample entry. The number of spectra is the product of the first two dimensions of `data` (for example, `(121,51,1167)` contains `121 x 51` spectra, each with 1167 channels, following the pre-processing done by Lita et al.)

In [10]:
records = pd.DataFrame([
    {
        "entry_index": i,
        "patient_id": item["patient_id"],
        "sample_name": item["name"],
        "class_id": item["class_id"],
        "n_spectra": int(np.prod(item["data"].shape[:-1])),
        "n_channels": item["data"].shape[-1],
    }
    for i, item in enumerate(dataset)
])

records.head()

,entry_index,patient_id,sample_name,class_id,n_spectra,n_channels
0,0,305,HF-305_v4b_1_1.h5_6,2,6171,1167
1,1,442,HF-442_V4BB_1.h5_12,4,2500,1167
2,2,448,HF-448_V5B_1.h5_3,1,3600,1167
3,3,592,HF-592_V3T_1.h5_4,6,3600,1167
4,4,615,HF-615_V5BB_1.h5_9,2,3600,1167


## Binary target mapping

In [11]:
target_by_class = {
    1: "IDHmutant",
    2: "IDHmutant",
    3: "IDHmutant",
    4: "IDHwildtype",
    5: "IDHwildtype",
    6: "IDHwildtype",
}

records["target"] = records["class_id"].map(target_by_class)
assert records["target"].notna().all(), "Unexpected or missing class_id found."

class_counts_per_patient = records.groupby("patient_id")["class_id"].nunique()
assert class_counts_per_patient.eq(1).all(), "At least one patient has multiple class_id values."

records.head()

,entry_index,patient_id,sample_name,class_id,n_spectra,n_channels,target
0,0,305,HF-305_v4b_1_1.h5_6,2,6171,1167,IDHmutant
1,1,442,HF-442_V4BB_1.h5_12,4,2500,1167,IDHwildtype
2,2,448,HF-448_V5B_1.h5_3,1,3600,1167,IDHmutant
3,3,592,HF-592_V3T_1.h5_4,6,3600,1167,IDHwildtype
4,4,615,HF-615_V5BB_1.h5_9,2,3600,1167,IDHmutant


## Summary at the patient level

In [20]:
patients = (
    records.groupby("patient_id", as_index=False)
    .agg(
        target=("target", "first"),
        n_samples=("entry_index", "size"),
        n_spectra=("n_spectra", "sum"),
    )
)

cohort_summary = (
    patients.groupby("target")
    .agg(
        patients=("patient_id", "nunique"),
        sample_entries=("n_samples", "sum"),
        spectra=("n_spectra", "sum"),
    )
    .reindex(["IDHmutant", "IDHwildtype"])
)

cohort_summary = cohort_summary.reset_index() # fixed misaligned table formatting
display(cohort_summary)
print(f"Unique patients: {len(patients)}")


,target,patients,sample_entries,spectra
0,IDHmutant,21,26,167414
1,IDHwildtype,25,33,133092


Unique patients: 46


In [13]:
print(cohort_summary.shape)
print(cohort_summary.index.tolist())
print(records["target"].value_counts(dropna=False))

(2, 3)
['IDHmutant', 'IDHwildtype']
target
IDHwildtype    33
IDHmutant      26
Name: count, dtype: int64


In [ ]:
N_SPLITS = 5
RANDOM_STATE = 42

patients_per_class = patients["target"].value_counts()
assert patients_per_class.min() >= N_SPLITS, ( # Set minimum num of patients to at least the number of splits
    "Requires at least five patients in each target group."
)

splitter = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

patients["fold"] = -1
for fold, (_, validation_positions) in enumerate(
    splitter.split(patients["patient_id"], patients["target"])
):
    patients.loc[validation_positions, "fold"] = fold

assert patients["fold"].between(0, N_SPLITS - 1).all()
patients.sort_values(["fold", "target", "patient_id"]).reset_index(drop=True)

,patient_id,target,n_samples,n_spectra,fold
0,305,IDHmutant,1,6171,0
1,868,IDHmutant,1,7021,0
2,1016,IDHmutant,1,7497,0
3,1887,IDHmutant,1,5796,0
4,2534,IDHmutant,1,4096,0
5,1043,IDHwildtype,1,4096,0
6,2102,IDHwildtype,3,12288,0
7,2454,IDHwildtype,1,4096,0
8,2802,IDHwildtype,1,4096,0
9,2898,IDHwildtype,1,4096,0


In [26]:
fold_rows = []
for fold in range(N_SPLITS):
    patient_fold = patients[patients["fold"] == fold]
    entry_fold = records[records["patient_id"].isin(patient_fold["patient_id"])]
    fold_rows.append({
        "fold": fold + 1,
        "validation_patients": len(patient_fold),
        "IDHmutant_patients": int((patient_fold["target"] == "IDHmutant").sum()),
        "IDHwildtype_patients": int((patient_fold["target"] == "IDHwildtype").sum()),
        "validation_sample_entries": len(entry_fold),
        "validation_spectra": int(entry_fold["n_spectra"].sum()),
    })

fold_summary = pd.DataFrame(fold_rows).set_index("fold")
display(fold_summary)

,validation_patients,IDHmutant_patients,IDHwildtype_patients,validation_sample_entries,validation_spectra
fold,,,,,
1,10,5,5,12,59253
2,9,4,5,11,54925
3,9,4,5,13,67396
4,9,4,5,11,57543
5,9,4,5,12,61389
